In [4]:
import numpy as np
from scipy.stats import norm

# ============================================================
# Paper code: Gaussian f-DP audit helpers
# ============================================================

def rh(inverse_blow_up_function, alpha, beta, j, m, k=2):
    h = [0.0 for _ in range(j + 1)]
    r = [0.0 for _ in range(j + 1)]
    h[j] = beta
    r[j] = alpha
    for i in range(j - 1, -1, -1):
        h[i] = max(h[i + 1], (k - 1) * inverse_blow_up_function(r[i + 1]))
        r[i] = r[i + 1] + (i / (m - i)) * (h[i] - h[i + 1])
    return r, h

def audit_rh(inverse_blow_up_function, m, c, threshold=0.05, k=2):
    alpha = threshold * c / m
    beta = threshold * (m - c) / m
    r, h = rh(inverse_blow_up_function, alpha, beta, c, m, k)
    return not (r[0] + h[0] > 1.0)

def rh_with_cap(inverse_blow_up_function, alpha, beta, j, m, c_cap, k=2):
    h = [0.0 for _ in range(j + 1)]
    r = [0.0 for _ in range(j + 1)]
    h[j] = beta
    r[j] = alpha
    for i in range(j - 1, -1, -1):
        h[i] = max(h[i + 1], (k - 1) * inverse_blow_up_function(r[i + 1]))
        r[i] = r[i + 1] + (i / (c_cap - i)) * (h[i] - h[i + 1])
    return r, h

def audit_rh_with_cap(inverse_blow_up_function, m, c, c_cap, threshold=0.05, k=2):
    threshold = threshold * c_cap / m
    alpha = threshold * c / c_cap
    beta = threshold * (c_cap - c) / c_cap
    r, h = rh_with_cap(inverse_blow_up_function, alpha, beta, c, m, c_cap, k)
    return not (r[0] + h[0] > c_cap / m)

def gaussianDP_blow_up_function(noise):
    def blow_up_function(x):
        threshold = norm.ppf(x)
        blown_up_threshold = threshold + 1.0 / noise
        return norm.cdf(blown_up_threshold)
    return blow_up_function

def gaussianDP_blow_up_inverse(noise):
    def blow_up_inverse_function(x):
        threshold = norm.ppf(x)
        blown_up_threshold = threshold - 1.0 / noise
        return norm.cdf(blown_up_threshold)
    return blow_up_inverse_function

def calculate_delta_gaussian(noise, epsilon):
    return norm.cdf(-epsilon * noise + 1.0 / (2.0 * noise)) - np.exp(epsilon) * norm.cdf(
        -epsilon * noise - 1.0 / (2.0 * noise)
    )

def calculate_epsilon_gaussian(noise, delta):
    epsilon_upper = 100.0
    epsilon_lower = 0.0
    while epsilon_upper - epsilon_lower > 1e-3:
        epsilon_middle = (epsilon_upper + epsilon_lower) / 2.0
        if calculate_delta_gaussian(noise, epsilon_middle) > delta:
            epsilon_lower = epsilon_middle
        else:
            epsilon_upper = epsilon_middle
    return epsilon_upper



In [5]:
from pathlib import Path

import numpy as np

results_root = Path("/u/bdkim4/bb-audit-dpsgd-renyi/exp_data/max_grad_norm/1.0")
run_name = "cifar10_half_cnn_eps2.0"
seeds = [0, 1, 2, 3, 4]

losses_in_by_seed = {}
losses_out_by_seed = {}

for seed in seeds:
    run_dir = results_root / f"seed{seed}" / run_name
    losses_in_by_seed[seed] = np.load(run_dir / "losses_in.npy")
    losses_out_by_seed[seed] = np.load(run_dir / "losses_out.npy")

losses_in = np.concatenate([losses_in_by_seed[seed] for seed in seeds])
losses_out = np.concatenate([losses_out_by_seed[seed] for seed in seeds])

print("loaded seeds:", seeds)
print("run_name:", run_name)
print("losses_in shape:", losses_in.shape)
print("losses_out shape:", losses_out.shape)


loaded seeds: [0, 1, 2, 3, 4]
run_name: cifar10_half_cnn_eps2.0
losses_in shape: (500,)
losses_out shape: (500,)


In [6]:
import numpy as np
from scipy.stats import norm

# Assumes the paper's functions already exist from the previous cell:
#   rh_with_cap
#   audit_rh_with_cap
#   gaussianDP_blow_up_inverse
#   calculate_epsilon_gaussian

# --------------------------
# user-tunable parameters
# --------------------------
delta = 1e-5       # delta at which to report epsilon
threshold = 0.05   # paper uses tau = 0.05
seed = 0

# Candidate Gaussian f-DP hypotheses (noise stds), low -> high privacy.
# If you hit a boundary status, widen this range.
candidate_noises = np.linspace(0.10, 8.0, 3000)

rng = np.random.default_rng(seed)

losses_in = np.asarray(losses_in, dtype=float).ravel()
losses_out = np.asarray(losses_out, dtype=float).ravel()
assert losses_in.ndim == 1 and losses_out.ndim == 1
assert len(losses_in) > 1 and len(losses_out) > 1

# We build one k=2 audit game with m coordinates from the two iid samples.
m = min(len(losses_in), len(losses_out))

# Scan a range of guess budgets (abstention levels) and keep the best one.
guess_grid = np.unique(
    np.r_[
        np.arange(5, min(50, m) + 1),
        np.linspace(max(5, min(60, m)), m, 40, dtype=int),
        m,
    ]
)
guess_grid = guess_grid[(guess_grid >= 1) & (guess_grid <= m)]

# ------------------------------------------------------------
# 1) Black-box attack score on scalar losses
#    Replace this with any better score if you already have one.
# ------------------------------------------------------------
# Simple parametric choice: log-likelihood ratio under Gaussian fits.
mu_in, std_in = losses_in.mean(), losses_in.std(ddof=1)
mu_out, std_out = losses_out.mean(), losses_out.std(ddof=1)
std_in = max(std_in, 1e-12)
std_out = max(std_out, 1e-12)

def loss_score(x):
    return norm.logpdf(x, loc=mu_in, scale=std_in) - norm.logpdf(x, loc=mu_out, scale=std_out)

# ------------------------------------------------------------
# 2) Build one Bernoulli(1/2) guessing game from the iid samples
# ------------------------------------------------------------
# u_i = 1 means the coordinate uses an "in" loss; u_i = 0 uses an "out" loss.
u = rng.integers(0, 2, size=m)

perm_in = rng.permutation(len(losses_in))
perm_out = rng.permutation(len(losses_out))

x = np.empty(m, dtype=float)
x[u == 1] = losses_in[perm_in[: np.sum(u == 1)]]
x[u == 0] = losses_out[perm_out[: np.sum(u == 0)]]

scores = loss_score(x)
pred = (scores >= 0).astype(int)   # guess "in" if score >= 0, else "out"
confidence = np.abs(scores)

# Sort once by confidence so c for any c_cap is just a prefix sum.
order = np.argsort(-confidence)
correct_sorted = (pred[order] == u[order]).astype(int)
cum_correct = np.cumsum(correct_sorted)

# Precompute inverse blow-up functions for the candidate Gaussian hypotheses.
inverse_blow_up_functions = [gaussianDP_blow_up_inverse(noise) for noise in candidate_noises]

def empirical_eps_from_counts(
    m, c, c_cap, candidate_noises, inverse_blow_up_functions, delta, threshold=0.05, k=2
):
    """
    Paper-style scan over Gaussian f-DP hypotheses:
    move from weak privacy (small noise / large eps) to strong privacy (large noise / small eps)
    and return the first hypothesis that the audit rejects.
    """
    idx = 0
    while idx < len(candidate_noises) and audit_rh_with_cap(
        inverse_blow_up_functions[idx], m=m, c=int(c), c_cap=int(c_cap), threshold=threshold, k=k
    ):
        idx += 1

    if idx == 0:
        # Already rejected at the smallest noise in the grid; extend the grid downward
        # if you want a tighter lower bound.
        return {
            "emp_eps": calculate_epsilon_gaussian(candidate_noises[0], delta),
            "emp_noise": candidate_noises[0],
            "status": "rejected_at_smallest_noise_expand_grid_down",
        }

    if idx == len(candidate_noises):
        # Nothing in the scanned range was rejected.
        return {
            "emp_eps": 0.0,
            "emp_noise": np.nan,
            "status": "no_rejection_in_grid",
        }

    return {
        "emp_eps": calculate_epsilon_gaussian(candidate_noises[idx], delta),
        "emp_noise": candidate_noises[idx],
        "status": "ok",
    }

# ------------------------------------------------------------
# 3) Evaluate all abstention levels and keep the best one
# ------------------------------------------------------------
results = []
for c_cap in guess_grid:
    c = int(cum_correct[c_cap - 1])
    out = empirical_eps_from_counts(
        m=m,
        c=c,
        c_cap=c_cap,
        candidate_noises=candidate_noises,
        inverse_blow_up_functions=inverse_blow_up_functions,
        delta=delta,
        threshold=threshold,
        k=2,
    )
    results.append(
        {
            "m": int(m),
            "c_cap": int(c_cap),
            "c": int(c),
            "attack_acc": c / c_cap,
            **out,
        }
    )

best = max(results, key=lambda r: r["emp_eps"])

print(f"m = {m}")
print(f"mean(losses_in)  = {mu_in:.6g}, std = {std_in:.6g}")
print(f"mean(losses_out) = {mu_out:.6g}, std = {std_out:.6g}")
print()
print("Best paper-style audit result")
print("-----------------------------")
print(f"delta              : {delta}")
print(f"guesses (c_cap)    : {best['c_cap']}")
print(f"correct guesses (c): {best['c']}")
print(f"attack accuracy    : {best['attack_acc']:.4f}")
print(f"empirical eps      : {best['emp_eps']:.6f}")
print(f"empirical noise    : {best['emp_noise']:.6f}")
print(f"status             : {best['status']}")

# Optional: inspect the top few abstention levels
top10 = sorted(results, key=lambda r: (-r["emp_eps"], r["c_cap"]))[:10]
top10

KeyboardInterrupt: 

In [ ]:
import numpy as np

# Standard RDP orders to report
orders = np.array([2], dtype=float)

def gaussian_fdp_noise_to_mu(noise):
    noise = float(noise)
    if noise <= 0:
        raise ValueError("noise must be > 0")
    return 1.0 / noise

def gaussian_fdp_noise_to_rdp(noise, orders):
    """
    Exact conversion for the Gaussian f-DP / GDP family used in the paper.
    If sigma=noise, then eps_RDP(alpha) = alpha / (2 sigma^2).
    """
    noise = float(noise)
    orders = np.asarray(orders, dtype=float)
    if noise <= 0:
        raise ValueError("noise must be > 0")
    if np.any(orders <= 1):
        raise ValueError("All RDP orders must be > 1")
    return orders / (2.0 * noise**2)

def rdp_to_eps_delta_bound(orders, eps_rdp, delta):
    """
    Standard RDP -> (eps, delta)-DP conversion:
      eps(delta) <= min_alpha eps_rdp(alpha) + log(1/delta)/(alpha-1)
    This is an upper bound. For the Gaussian case, the exact GDP epsilon
    from calculate_epsilon_gaussian(...) is already available as best['emp_eps'].
    """
    orders = np.asarray(orders, dtype=float)
    eps_rdp = np.asarray(eps_rdp, dtype=float)
    delta = float(delta)
    if delta <= 0 or delta >= 1:
        raise ValueError("delta must be in (0, 1)")
    vals = eps_rdp + np.log(1.0 / delta) / (orders - 1.0)
    idx = int(np.argmin(vals))
    return float(vals[idx]), float(orders[idx])

# Use the same audited Gaussian f-DP hypothesis returned by the previous cell.
emp_noise = float(best["emp_noise"])
if not np.isfinite(emp_noise):
    raise ValueError(f"best['emp_noise'] is not finite; status={best.get('status')}")

mu_gdp = gaussian_fdp_noise_to_mu(emp_noise)
eps_rdp = gaussian_fdp_noise_to_rdp(emp_noise, orders)
eps_delta_from_rdp_bound, best_order = rdp_to_eps_delta_bound(orders, eps_rdp, delta)

fdp_rdp_summary = {
    "emp_noise": emp_noise,                  # sigma of audited Gaussian f-DP curve
    "mu_gdp": mu_gdp,                        # equivalent GDP parameter
    "orders": orders,
    "eps_rdp": eps_rdp,
    "eps_delta_exact_from_gdp": float(best["emp_eps"]),
    "eps_delta_upper_bound_from_rdp": eps_delta_from_rdp_bound,
    "best_order_for_rdp_bound": best_order,
    "audit_status": best.get("status"),
    "c_cap": int(best["c_cap"]),
    "c": int(best["c"]),
    "attack_acc": float(best["attack_acc"]),
}

print("Audited Gaussian f-DP -> RDP")
print("----------------------------")
print(f"audited sigma (noise)      : {fdp_rdp_summary['emp_noise']:.6f}")
print(f"equivalent mu-GDP          : {fdp_rdp_summary['mu_gdp']:.6f}")
print(f"exact eps(delta) from GDP  : {fdp_rdp_summary['eps_delta_exact_from_gdp']:.6f}   (delta={delta})")
print(f"RDP->DP upper bound        : {fdp_rdp_summary['eps_delta_upper_bound_from_rdp']:.6f}   (best alpha={fdp_rdp_summary['best_order_for_rdp_bound']})")
print()
print("RDP curve:")
for a, e in zip(fdp_rdp_summary["orders"], fdp_rdp_summary["eps_rdp"]):
    print(f"  alpha={a:>6g}   eps_RDP={e:.6f}")

fdp_rdp_summary